# Objetivo
Refatorar os extractores e transformadores de robocalls para aplicação universal. 

Neste notebook serão testadas, sobre uma amostra dos dados de CDRs, funções para:

- extrair CDRs processados do formato texto para parquet;
- transformar CDRs para formato normalizado (tabela única com campos uniformes);

> Atenção: este notebook deve ser executado no kernel com Python 3.9

# Configuração do ambiente

## Bibliotecas

In [ ]:
import logging

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from teleutils import robocalls
from teleutils.core.extractors import CDRTextExtractor
from teleutils.core.transformers import CDRTransformer

## Logging

In [ ]:
# Configuração mínima para exibir logs no output da célula
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)

# Opcional: reduzir ruído de bibliotecas externas (pyspark, py4j, etc.)
logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("pyspark").setLevel(logging.WARNING)

## Spark Session

### Local

In [ ]:
spark = SparkSession.builder \
    .master("local[1]") \
    .appName("testes_chamadas_abusivas") \
    .config("spark.executor.memory", "512m") \
    .getOrCreate()
spark

## Pastas origem e destino

In [ ]:
# Pasta base origem
INPUT_FOLDER = "/data/cdr/chamadas_abusivas/cdr_processado/Semana86"

# Pasta de arquivos extraídos
EXTRACTED_FOLDER = "/data/cdr/chamadas_abusivas/cdr_extraido/Semana86"

# Pasta de arquivos transformados
TRANSFORMED_FOLDER = "/data/cdr/chamadas_abusivas/cdr_transformado/Semana86"

## Caminho completo dos arquivos

### Pastas locais

In [ ]:
INPUT_FOLDER = "/mnt/e/data/datasets/amostra_cdr/semana95"
EXTRACTED_FOLDER = "/home/maxwelfreitas/datasets/cdr_extraido"
TRANSFORMED_FOLDER = "/home/maxwelfreitas/datasets/cdr_transformado"

### Caminho os arquivos

In [ ]:
# Claro/Ericsson
cdr_processado_claro_ericsson = f"{INPUT_FOLDER}/claro/ericsson"
cdr_extraido_claro_ericsson = f"{EXTRACTED_FOLDER}/claro_ericsson_extraido.parquet"
cdr_transformado_claro_ericsson = f"{TRANSFORMED_FOLDER}/claro_ericsson_transformado.parquet"

# Claro/Nokia
cdr_processado_claro_nokia = f"{INPUT_FOLDER}/claro/nokia"
cdr_extraido_claro_nokia = f"{EXTRACTED_FOLDER}/claro_nokia_extraido.parquet"
cdr_transformado_claro_nokia = f"{TRANSFORMED_FOLDER}/claro_nokia_transformado.parquet"

# Tim/Ericsson
cdr_processado_tim_ericsson = f"{INPUT_FOLDER}/tim/ericsson"
cdr_extraido_tim_ericsson = f"{EXTRACTED_FOLDER}/tim_ericsson_extraido.parquet"
cdr_transformado_tim_ericsson = f"{TRANSFORMED_FOLDER}/tim_ericsson_transformado.parquet"
cdr_ofensores_tim_ericsson = f"{TRANSFORMED_FOLDER}/tim_ericsson_ofensores.parquet"

# Vivo/Ericsson
cdr_processado_vivo_ericsson = f"{INPUT_FOLDER}/vivo/ericsson"
cdr_extraido_vivo_ericsson = f"{EXTRACTED_FOLDER}/vivo_ericsson_extraido.parquet"
cdr_transformado_vivo_ericsson = f"{TRANSFORMED_FOLDER}/vivo_ericsson_transformado.parquet"
cdr_ofensores_vivo_ericsson = f"{TRANSFORMED_FOLDER}/vivo_ericsson_ofensores.parquet"

# Tim/ATS
cdr_processado_tim_ats = f"{INPUT_FOLDER}/tim/ats"
cdr_extraido_tim_ats = f"{EXTRACTED_FOLDER}/tim_ats_extraido.parquet"
cdr_transformado_tim_ats = f"{TRANSFORMED_FOLDER}/tim_ats_transformado.parquet"
cdr_ofensores_tim_ats = f"{TRANSFORMED_FOLDER}/tim_ats_ofensores.parquet"

# Vivo/FCDR
cdr_processado_vivo_fcdr = f"{INPUT_FOLDER}/vivo/fcdr"
cdr_extraido_vivo_fcdr = f"{EXTRACTED_FOLDER}/vivo_fcdr_extraido.parquet"
cdr_transformado_vivo_fcdr = f"{TRANSFORMED_FOLDER}/vivo_fcdr_transformado.parquet"
cdr_ofensores_vivo_fcdr = f"{TRANSFORMED_FOLDER}/vivo_fcdr_ofensores.parquet"

# Testes

## Extração

In [ ]:
extractor = CDRTextExtractor(spark)
extractor

### Ericsson

In [ ]:
df_extraido_claro_ericsson = extractor.extract_cdr_ericsson(cdr_processado_claro_ericsson, cdr_extraido_claro_ericsson)
df_extraido_claro_ericsson.show(5)

In [ ]:
df_extraido_tim_ericsson = extractor.extract_cdr_ericsson(cdr_processado_tim_ericsson, cdr_extraido_tim_ericsson)
df_extraido_tim_ericsson.show(5)

In [ ]:
df_extraido_vivo_ericsson = extractor.extract_cdr_ericsson(cdr_processado_vivo_ericsson, cdr_extraido_vivo_ericsson)
df_extraido_vivo_ericsson.show(5)

### Tim/ATS

In [ ]:
df_extraido_tim_ats = extractor.extract_cdr_tim_ats(cdr_processado_tim_ats,cdr_extraido_tim_ats)
df_extraido_tim_ats.show(5)

### Vivo/FCDR

In [ ]:
df_extraido_vivo_fcdr = extractor.extract_cdr_vivo_fcdr(cdr_processado_vivo_fcdr,cdr_extraido_vivo_fcdr)
df_extraido_vivo_fcdr.show(5)

### Claro/Nokia

In [ ]:
df_extraido_claro_nokia = extractor.extract_cdr_claro_nokia(cdr_processado_claro_nokia,cdr_extraido_claro_nokia)
df_extraido_claro_nokia.show(5)

## Transformação

In [ ]:
transformer = CDRTransformer(spark)
transformer

### Ericsson

In [ ]:
df = transformer.transform_cdr_ericsson(cdr_extraido_claro_ericsson, cdr_transformado_claro_ericsson)
df.show(5)
df.printSchema()

In [ ]:
df.groupBy("no_tipo_chamada").count().show()

In [ ]:
df = transformer.transform_cdr_ericsson(cdr_extraido_tim_ericsson,cdr_transformado_tim_ericsson)
df.show(5)
df.printSchema()

In [ ]:
df = transformer.transform_cdr_ericsson(cdr_extraido_vivo_ericsson,cdr_transformado_vivo_ericsson)
df.show(5)
df.printSchema()

### Claro/Nokia

In [ ]:
df = transformer.transform_cdr_claro_nokia(cdr_extraido_claro_nokia, cdr_transformado_claro_nokia)
df.show(5)
df.printSchema()

### Tim/ATS

In [ ]:
df = transformer.transform_cdr_tim_ats(cdr_extraido_tim_ats,cdr_transformado_tim_ats)
df.show(5)
df.printSchema()

In [ ]:
df.groupBy("no_autenticacao").agg(F.count("nu_referencia")).show()

In [ ]:
df.groupBy("no_autenticacao").pivot("no_tipo_chamada").agg(F.count("nu_referencia")).show()

### Vivo/FCDR

In [ ]:
df_transformado_vivo_fcdr = transformer.transform_cdr_vivo_fcdr(cdr_extraido_vivo_fcdr,cdr_transformado_vivo_fcdr)
df_transformado_vivo_fcdr.show(5)
df_transformado_vivo_fcdr.printSchema()

In [ ]:
df_transformado_vivo_fcdr.groupBy("no_autenticacao").agg(F.count("nu_referencia")).show(truncate=False)

In [ ]:
df_transformado_vivo_fcdr.groupBy("no_autenticacao").pivot("no_tipo_chamada").agg(F.count("nu_referencia")).show(truncate=False)